# 06 · AI-ECG age gap and death within 1 year

Does the retest variation of AI-ECG age weaken its association with mortality, and does averaging repeat ECGs help? For each of the ten
models, this notebook

1. estimates the odds ratio (OR) for death within 1 year per 10-year larger age gap of the first ECG, adjusted for age, sex and the number
   of ECGs within 48 hours (Supplementary Table 4);
2. estimates the OR without retest variation by simulation–extrapolation (SIMEX), and the share of the association hidden by the variation
   (Fig. 4a, Supplementary Table 4);
3. compares the OR of the mean age gap of the first one, two, three and five of five repeat ECGs with that of each ECG alone, for five ECGs
   within 7 days and within 48 hours (Fig. 4b, Supplementary Fig. 6, Supplementary Table 4);
4. gives the share of patients whose 'older' flag (age gap of 8 years or more) differs between the first ECG and the next ECG within 48
   hours (Fig. 4c).

| Input | Made in | Used for |
|---|---|---|
| `PREDICTIONS/{key}.parquet` | notebook 04 | AI-ECG age of the included ECGs of the test split |
| `MIMIC_RECORDS`, `COHORT`, `ECG_CONTEXT` | notebooks 01, 02 | patient of each ECG; inclusion and split; age, sex, death within 1 year |
| `MORTALITY_COHORT` | notebook 02 | per test patient with known sex: the first ECG followed by another ECG within 48 hours, and that next ECG |
| `AVERAGING_7D`, `AVERAGING_48H` | notebook 02 | per test patient: the first run of five ECGs within 7 days, or within 48 hours |

The results go to `SUMMARY` (`mortality.csv`, `simex_curve.csv`, `averaging.csv`, `flags.csv`), from which notebooks 07 and 08 build the
tables and figures. Every OR is per `OR_PER` = 10-year larger age gap. Each patient contributes one row to each cohort, so the regressions
are unweighted, and the 95% confidence intervals are percentile intervals from `N_BOOT` bootstrap resamples of patients.

In [ ]:
%run ./00_setup_and_settings.ipynb

## What to run

The analyses of each model are written to their own files under `SUMMARY/by_model/`. A file that exists is not computed again, so an
interrupted run can be resumed; delete the files of a model to compute it again. The tables in `SUMMARY` are assembled at the end of the
notebook, once every model is done.

SIMEX takes most of the time: about 100,000 logistic regressions per model with 1,000 resamples, about 12 minutes per model on a desktop
CPU (the averaging and the flag take less than half a minute), so about 2 hours for the ten models. For a quick functional check, set
`RUN_KEYS = ["lima"]` and lower `N_BOOT` in notebook 00 (for example to 50), with `SUMMARY` pointing to another folder.

In [ ]:
# ============================== WHAT TO RUN ==============================
RUN_KEYS = MODEL_KEYS              # the models to analyse (all ten)
PARTS = SUMMARY / "by_model"       # one file per analysis and model

In [ ]:
import time

PARTS.mkdir(parents=True, exist_ok=True)

# SIMEX: noise is added to the age gap of the first ECG with a variance of lambda times the within-patient variance of the gap
SIMEX_LAMBDAS = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 3.0])      # 0 = the observed data; the curve is extrapolated to lambda = -1
SIMEX_REPS = 100                                              # noise replicates per level for the estimate
SIMEX_REPS_BOOT = 20                                          # noise replicates per level in each bootstrap resample
# number of ECGs of the patient from the first ECG of the cohort (`n_ecgs`, notebook 02): lower bounds of the categories
COUNT_BOUNDS = {"mortality": [2, 3, 4, 6],                    # 2, 3, 4-5, >=6 ECGs within 48 h
                "within_7d": [5, 7, 10, 15],                  # 5-6, 7-9, 10-14, >=15 ECGs within 7 days
                "within_48h": [5, 7, 10]}                     # 5-6, 7-9, >=10 ECGs within 48 h
AVERAGED = [1, 2, 3, 5]                                       # mean age gap of the first k ECGs of the run of five

## ECGs of the test split

The analyses use the included ECGs of the test split with known sex. The outcome of a cohort is death within 1 year (0–365 days) after its
first ECG.

In [ ]:
rec = pd.read_parquet(MIMIC_RECORDS, columns=["idx", "subject_id"]).set_index("idx")
coh = pd.read_parquet(COHORT, columns=["idx", "include", "split"]).set_index("idx")
ctx = pd.read_parquet(ECG_CONTEXT, columns=["idx", "age", "sex", "death_1y"]).set_index("idx")

ecg = rec.loc[coh.index[coh["include"].astype(bool) & (coh["split"] == "test")]].join(ctx)
ecg = ecg[ecg["sex"].notna()]
ecg["male"] = (ecg["sex"] == "M").astype(float)
print(f"test split: {masked(len(ecg))} ECGs of {masked(ecg['subject_id'].nunique())} patients with known sex")

## Cohorts

- **Mortality cohort**: for each test patient with known sex, the first ECG followed by another ECG within 48 hours (`idx_first`) and that
  next ECG (`idx_next`). The number of ECGs of the patient within 48 hours from the first ECG (2, 3, 4–5, 6 or more) enters the models as
  a marker of acuity.
- **Averaging cohorts**: test patients with five or more ECGs within 7 days, or within 48 hours, with their first such run of five ECGs
  (`idx_1` … `idx_5`); the number of ECGs is counted in the same window from the first ECG of the run.

The number of ECGs (`n_ecgs`, stored with each cohort by notebook 02) includes the first ECG; ECGs recorded at the same time as the first ECG
but listed before it are not counted. Rows are put in the order of `subject_id`, so that the bootstrap resamples do not depend on the order
of the cohort files.

In [ ]:
def count_categories(counts, bounds):
    """Indicators of the categories of the number of ECGs (the lowest category is the reference) and the patients per category."""
    k = np.clip(np.searchsorted(bounds, counts, side="right") - 1, 0, len(bounds) - 1)
    labels = [f"{a}" if b - a == 1 else f"{a}-{b - 1}" for a, b in zip(bounds[:-1], bounds[1:])] + [f">={bounds[-1]}"]
    Z = np.column_stack([(k == j).astype(float) for j in range(1, len(bounds))])
    return Z, dict(zip(labels, np.bincount(k, minlength=len(bounds)).tolist()))


def cohort(path, idx_cols, name):
    c = pd.read_parquet(path).sort_values("subject_id").reset_index(drop=True)
    for col in idx_cols:
        assert c[col].isin(ecg.index).all(), "a cohort ECG is not among the ECGs of the test split with known sex"
    first = ecg.loc[c[idx_cols[0]].to_numpy()]
    Z, sizes = count_categories(c["n_ecgs"].to_numpy(), COUNT_BOUNDS[name])
    return {"subject_id": c["subject_id"].to_numpy(), "idx": [c[col].to_numpy() for col in idx_cols], "Z": Z, "sizes": sizes,
            "age": first["age"].to_numpy(float), "male": first["male"].to_numpy(float), "y": first["death_1y"].astype(bool).to_numpy(int)}


COHORTS = {"mortality": cohort(MORTALITY_COHORT, ["idx_first", "idx_next"], "mortality"),
           "within_7d": cohort(AVERAGING_7D, [f"idx_{j}" for j in range(1, 6)], "within_7d"),
           "within_48h": cohort(AVERAGING_48H, [f"idx_{j}" for j in range(1, 6)], "within_48h")}
for name, C in COHORTS.items():
    print(f"{name:10s}: {masked(len(C['y']))} patients, {masked(C['y'].sum())} deaths within 1 year ({100 * C['y'].mean():.1f}%); "
          f"ECGs in the window: " + ", ".join(f"{lab}: {masked(n)}" for lab, n in C["sizes"].items()))

## Logistic regression

Death within 1 year is regressed on age (in decades, centred at 65 years), sex, the indicators of the number of ECGs and the age gap
divided by `OR_PER`, so that exp(coefficient of the gap) is the OR per 10-year larger age gap. The models are unpenalised and fitted by
Newton–Raphson iterations until no coefficient changes by more than 10⁻⁹.

In [ ]:
def design(age, male, gap, Z=None):
    return np.column_stack([(age - 65) / 10, male] + ([Z] if Z is not None else []) + [gap / OR_PER])


def fit_logistic(X, y, start=None, max_iter=40):
    """Coefficients (intercept first) of an unpenalised logistic regression, by Newton-Raphson from `start` (default: zeros)."""
    Xc = np.column_stack([np.ones(len(y)), X])
    b = np.zeros(Xc.shape[1]) if start is None else np.array(start, float)
    for _ in range(max_iter):
        p = 1 / (1 + np.exp(-(Xc @ b)))
        step = np.linalg.solve((Xc * (p * (1 - p))[:, None]).T @ Xc, Xc.T @ (y - p))
        b += step
        if np.abs(step).max() < 1e-9:
            break
    return b


def gaps_of(pred_age):
    """Age gap (AI-ECG age minus chronological age) of every ECG of the analysis, indexed by idx."""
    return pd.Series(pred_age.reindex(ecg.index).to_numpy().astype(float) - ecg["age"].to_numpy(), index=ecg.index.to_numpy())

## SIMEX

Retest variation in the age gap of the first ECG weakens its estimated association with death (regression dilution). SIMEX (Cook and
Stefanski) measures this weakening by adding more of the same variation and extrapolating back:

1. The within-patient variance of the age gap is estimated from the two ECGs of the mortality cohort as half the variance of their
   difference, σ²_w = Var(gap_next − gap_first) / 2.
2. Normal noise with variance λ σ²_w, λ = 0.5, 1, 1.5, 2 and 3, is added to the gap of the first ECG, and the model is refitted. At each
   level the log OR of the gap is averaged over 100 noise replicates (20 in each bootstrap resample). The noise multiplier 1 + λ is the
   total retest variance relative to the observed one (1 = observed data).
3. A quadratic in λ is fitted to the mean log ORs at λ = 0 (observed) to 3 and extrapolated to λ = −1, the point at which the retest
   variation already in the data would be removed.

The share of the association hidden by the variation is 1 − (observed log OR) / (extrapolated log OR). σ²_w is held fixed in the
bootstrap. The same estimates without the number of ECGs (age and sex only) are computed as point estimates. The noise replicates start
the Newton–Raphson iterations at the estimate of the data without added noise, so that fewer iterations are needed; because the
log-likelihood of the logistic model has a single maximum, the starting values do not change the estimates.

In [ ]:
def simex(C, ga, gb, rng):
    """Observed and SIMEX ORs in cohort C, from the age gaps of the first (ga) and of the next ECG (gb)."""
    sigma_w = float(np.sqrt(np.var(gb - ga) / 2))
    age, male, Z, y = C["age"], C["male"], C["Z"], C["y"]

    def curve(rows, reps, adjust=True):
        """Mean log OR at every noise level and its extrapolation to no retest variation, in the patients `rows`."""
        Zr = Z[rows] if adjust else None
        b_obs = fit_logistic(design(age[rows], male[rows], ga[rows], Zr), y[rows])
        logor = []
        for lam in SIMEX_LAMBDAS:
            if lam == 0:
                logor.append(b_obs[-1])
                continue
            reps_b = [fit_logistic(design(age[rows], male[rows], ga[rows] + np.sqrt(lam) * sigma_w * rng.standard_normal(len(rows)), Zr),
                                   y[rows], start=b_obs)[-1] for _ in range(reps)]
            logor.append(np.mean(reps_b))
        logor = np.array(logor)
        return logor, float(np.polyval(np.polyfit(SIMEX_LAMBDAS, logor, 2), -1.0))

    def hidden(b_obs, b_simex):
        return 100 * (1 - b_obs / b_simex) if b_simex > 0 else np.nan

    every = np.arange(len(y))
    logor, b_simex = curve(every, SIMEX_REPS)
    logor_as, b_simex_as = curve(every, SIMEX_REPS, adjust=False)         # age and sex only
    boot = np.array([(lb[0], sb) for lb, sb in (curve(r, SIMEX_REPS_BOOT) for r in bootstrap_patients(C["subject_id"], N_BOOT, rng))])
    row = {"patients": int(len(y)), "deaths": int(y.sum())}
    for name, value, values in (("or_obs", np.exp(logor[0]), np.exp(boot[:, 0])), ("or_simex", np.exp(b_simex), np.exp(boot[:, 1])),
                                ("hidden_pct", hidden(logor[0], b_simex), [hidden(a, b) for a, b in boot])):
        row[name] = float(value)
        row[f"{name}_lo"], row[f"{name}_hi"] = percentile_ci(values)
    row.update(or_obs_age_sex=float(np.exp(logor_as[0])), or_simex_age_sex=float(np.exp(b_simex_as)),
               hidden_pct_age_sex=float(hidden(logor_as[0], b_simex_as)))
    row.update({f"count_{lab.replace('>=', 'ge').replace('-', '_')}": n for lab, n in C["sizes"].items()})
    curve_rows = [{"noise_multiplier": 1 + lam, "or_per10": float(np.exp(b)), "extrapolated": False} for lam, b in zip(SIMEX_LAMBDAS, logor)]
    curve_rows.append({"noise_multiplier": 0.0, "or_per10": float(np.exp(b_simex)), "extrapolated": True})
    print(f"  within-patient SD of the age gap {sigma_w:.2f} years; OR observed {row['or_obs']:.3f} "
          f"({row['or_obs_lo']:.3f} to {row['or_obs_hi']:.3f}), SIMEX {row['or_simex']:.3f} ({row['or_simex_lo']:.3f} to "
          f"{row['or_simex_hi']:.3f}), hidden {row['hidden_pct']:.1f}%")
    return {"simex": [row], "simex_curve": curve_rows}

## Averaging of repeat ECGs

In each averaging cohort, the OR is estimated for the mean age gap of the first k ECGs of the run (k = 1, 2, 3, 5) and, as a control for
later ECGs being more informative, for each of the five ECGs alone; age and sex are those at the first ECG of the run. CIs are computed for
the means. The OR ratio compares the mean of five ECGs within 7 days with the first ECG, exp(β₅ − β₁), with its CI from the same resamples.

In [ ]:
def averaging(C, gaps, rng, with_ratio=False):
    """ORs of the mean age gaps and of each ECG alone in cohort C; with_ratio: also the OR ratio of the mean of five ECGs to the first ECG."""
    G = np.column_stack(gaps)
    age, male, Z, y = C["age"], C["male"], C["Z"], C["y"]
    series = {("mean", k): G[:, :k].mean(1) for k in AVERAGED}
    series.update({("single", j + 1): G[:, j] for j in range(G.shape[1])})
    b = {s: fit_logistic(design(age, male, g, Z), y)[-1] for s, g in series.items()}
    means = [s for s in series if s[0] == "mean"]
    boot = {s: [] for s in means}
    for r in bootstrap_patients(C["subject_id"], N_BOOT, rng):
        if y[r].sum() in (0, len(r)):                    # a resample without deaths or without survivors cannot be fitted
            continue
        for s in means:
            boot[s].append(fit_logistic(design(age[r], male[r], series[s][r], Z[r]), y[r])[-1])
    boot = {s: np.array(v) for s, v in boot.items()}
    rows = []
    for s in series:
        lo, hi = percentile_ci(np.exp(boot[s])) if s in boot else (np.nan, np.nan)
        rows.append({"kind": s[0], "k": s[1], "or_per10": float(np.exp(b[s])), "or_per10_lo": lo, "or_per10_hi": hi})
    if not with_ratio:
        return rows, None
    top = ("mean", max(AVERAGED))
    return rows, {"ratio": float(np.exp(b[top] - b[("mean", 1)])), "ratio_ci": percentile_ci(np.exp(boot[top] - boot[("mean", 1)]))}

## 'Older' flag

Lima et al. flagged accelerated aging when AI-ECG age exceeded chronological age by `OLDER_FLAG` = 8 years or more. In the mortality
cohort, we count the patients whose flag differs between the first ECG and the next ECG within 48 hours.

In [ ]:
def older_flag(C, ga, gb, rng):
    differs = (ga >= OLDER_FLAG) != (gb >= OLDER_FLAG)
    lo, hi = percentile_ci([100 * differs[r].mean() for r in bootstrap_patients(C["subject_id"], N_BOOT, rng)])
    return {"flags": [{"patients": int(len(differs)), "differs_pct": 100 * float(differs.mean()), "differs_pct_lo": lo, "differs_pct_hi": hi}]}

## Run

Each analysis of each model has its own random generator, derived from `BOOT_SEED` and the name of the analysis (`rng_for`); in SIMEX the
same generator draws the bootstrap resamples and the added noise.

In [ ]:
def run_parts(names, key, compute):
    """Compute the analyses `names` of one model and write them to PARTS, unless their files exist."""
    files = {n: PARTS / f"{n}_{key}.csv" for n in names}
    if all(f.exists() for f in files.values()):
        print(f"  {', '.join(names)}: exist, kept")
        return
    t0 = time.time()
    for n, rows in compute().items():
        if not files[n].exists():
            df = pd.DataFrame(rows)
            df.insert(0, "key", key)
            df.to_csv(files[n], index=False)
    print(f"  {', '.join(names)}: {(time.time() - t0) / 60:.1f} min")


def averaging_both(gap, key):
    rows, summary = [], {}
    for name, col in (("within_7d", "avg7d"), ("within_48h", "avg48h")):
        C = COHORTS[name]
        r, ratio = averaging(C, [gap.loc[i].to_numpy() for i in C["idx"]], rng_for(f"averaging {key} {name}"),
                             with_ratio=(name == "within_7d"))
        rows += [{"cohort": name, **x} for x in r]
        summary.update({f"{col}_patients": int(len(C["y"])), f"{col}_deaths": int(C["y"].sum())})
        if ratio is not None:
            summary.update(ratio_7d=ratio["ratio"], ratio_7d_lo=ratio["ratio_ci"][0], ratio_7d_hi=ratio["ratio_ci"][1])
    print(f"  OR ratio, mean of five ECGs within 7 days to the first ECG: {summary['ratio_7d']:.3f} "
          f"({summary['ratio_7d_lo']:.3f} to {summary['ratio_7d_hi']:.3f})")
    return {"averaging": rows, "averaging_summary": [summary]}


M = COHORTS["mortality"]
for key in RUN_KEYS:                       # about 12 minutes per model with the default settings, mostly SIMEX
    print(f"{key} ({model_label(key)})")
    gap = gaps_of(load_predictions(key).set_index("idx")["pred_age"])
    assert gap.notna().all(), "an ECG of the analysis has no AI-ECG age"
    g_first, g_next = gap.loc[M["idx"][0]].to_numpy(), gap.loc[M["idx"][1]].to_numpy()
    run_parts(["simex", "simex_curve"], key, lambda: simex(M, g_first, g_next, rng_for(f"simex {key}")))
    run_parts(["averaging", "averaging_summary"], key, lambda: averaging_both(gap, key))
    run_parts(["flags"], key, lambda: older_flag(M, g_first, g_next, rng_for(f"older flag {key}")))

## Results

The tables in `SUMMARY` are assembled from the files of the ten models, in the order of `MODEL_KEYS`. A table that exists already is kept;
delete it to assemble it again. `mortality.csv` joins the SIMEX results with the OR ratio and the sizes of the averaging cohorts; its last
columns give the patients per category of the number of ECGs within 48 hours.

In [ ]:
def read_parts(name):
    missing = [k for k in MODEL_KEYS if not (PARTS / f"{name}_{k}.csv").exists()]
    if missing:
        print(f"{name}: not assembled yet, missing {missing}")
        return None
    return pd.concat([pd.read_csv(PARTS / f"{name}_{k}.csv") for k in MODEL_KEYS], ignore_index=True)


def write_summary(name, df):
    out = SUMMARY / f"{name}.csv"
    if df is None:
        return None
    if out.exists():
        print(f"{name}.csv: exists, kept")
    else:
        df.to_csv(out, index=False)
        print(f"{name}.csv: {len(df)} rows")
    return df


MORTALITY_COLUMNS = ["key", "patients", "deaths", "or_obs", "or_obs_lo", "or_obs_hi", "or_simex", "or_simex_lo", "or_simex_hi",
                     "hidden_pct", "hidden_pct_lo", "hidden_pct_hi", "or_obs_age_sex", "or_simex_age_sex", "hidden_pct_age_sex",
                     "ratio_7d", "ratio_7d_lo", "ratio_7d_hi", "avg7d_patients", "avg7d_deaths", "avg48h_patients", "avg48h_deaths"]
sx, av = read_parts("simex"), read_parts("averaging_summary")
mortality = None
if sx is not None and av is not None:
    mortality = sx.merge(av, on="key", how="left")
    mortality = mortality[MORTALITY_COLUMNS + [c for c in mortality.columns if c.startswith("count_")]]
tables = {"mortality": write_summary("mortality", mortality),
          "simex_curve": write_summary("simex_curve", read_parts("simex_curve")),
          "averaging": write_summary("averaging", read_parts("averaging")),
          "flags": write_summary("flags", read_parts("flags"))}

In [ ]:
# Overview (aggregate model metrics)
if tables["mortality"] is not None and tables["flags"] is not None:
    overview = tables["mortality"].set_index("key")[["or_obs", "or_simex", "hidden_pct", "or_obs_age_sex", "ratio_7d"]].join(
        tables["flags"].set_index("key")["differs_pct"].rename("older flag differs %"))
    overview.index = [model_label(k) for k in overview.index]
    display(overview.round(3))